In [ ]:
# Noise-Aware QAOA – Examples

# Cell 1: Imports & seed
import numpy as np
import pandas as pd
from noise_aware_qaoa import (
    set_seed,
    run_suite_maxcut,
    export_artifacts,
    plot_approx_ratio,
)

set_seed(42)

# Cell 2: Minimal Max-Cut benchmark (p = 1,2,3)
bundle_min = run_suite_maxcut(
    n=8,
    p_list=[1, 2, 3],
    graph_p=0.4,
    shots=1024,
    # default ablations:
    #   warm_start=False → True
    #   optimizer=adam / nelder-mead
    #   readout_mit=False/True
    #   zne_scales=[1] or [1,3,5]
)
df_min = export_artifacts(bundle_min, "qaoa_min.csv", "qaoa_min.json")
plot_approx_ratio(df_min, "qaoa_min_ratio_vs_p.png")

# Cell 3: Custom ablations (explicit) for a clearer A/B compare
ablations = [
    {"warm_start": False, "mixer": "x", "optimizer": "adam",        "readout_mit": False, "zne_scales": [1]},
    {"warm_start": True,  "mixer": "x", "optimizer": "adam",        "readout_mit": True,  "zne_scales": [1, 3, 5]},
    {"warm_start": True,  "mixer": "x", "optimizer": "nelder-mead", "readout_mit": True,  "zne_scales": [1]},
]
bundle_ablate = run_suite_maxcut(
    n=10, p_list=[1, 2, 3, 4], graph_p=0.5, shots=1024,
    ablations=ablations, seed=42
)
df_ablate = export_artifacts(bundle_ablate, "qaoa_ablate.csv", "qaoa_ablate.json")
plot_approx_ratio(df_ablate, "qaoa_ablate_ratio_vs_p.png")

# Cell 4: Quick pandas peeks
print(df_ablate.head())
print("\nMean approx. ratio by p (using mitigated if available):")
tmp = df_ablate.copy()
tmp["approx_used"] = tmp["approx_ratio_mitig"].fillna(tmp["approx_ratio_plain"])
print(tmp.groupby("p")["approx_used"].mean())

print("\nWarm-start vs no warm-start (mean approx. ratio):")
print(tmp.groupby("warm_start")["approx_used"].mean())

# Cell 5: Noise sweep & ZNE toggle demo
for p1 in [0.0, 1e-3, 3e-3, 5e-3]:
    bundle_ns = run_suite_maxcut(
        n=8, p_list=[1, 2], graph_p=0.4, shots=1024,
        ablations=[{"warm_start": True, "mixer":"x", "optimizer":"adam", "readout_mit": True, "zne_scales":[1,3,5]}],
        noise_p1=p1, noise_p2=5*p1, readout_p=0.02, seed=123
    )
    df_ns = export_artifacts(bundle_ns, f"qaoa_noise_{p1:.4f}.csv", f"qaoa_noise_{p1:.4f}.json")
    print(f"Noise p1={p1:.4f} → mean approx ratio:",
          df_ns["approx_ratio_mitig"].fillna(df_ns["approx_ratio_plain"]).mean())

# Cell 6: Layout/Transpile metrics (depth, CX)
# (stored in the JSON bundle under 'layout')
import json
with open("qaoa_ablate.json") as f:
    J = json.load(f)
layout_df = pd.DataFrame(J["layout"])
print("\nTranspile/layout metrics:")
print(layout_df.groupby(["p", "mixer", "warm_start"])[["depth", "cx", "qubits"]].mean())

# Cell 7: Reproducibility sanity check
set_seed(99)
A = run_suite_maxcut(n=6, p_list=[1], shots=256, seed=99)
set_seed(99)
B = run_suite_maxcut(n=6, p_list=[1], shots=256, seed=99)
print("\nReproducibility (records count equal?):", len(A["records"]) == len(B["records"]))

# Cell 8: Batch graph sizes (concise)
sizes = [6, 8, 10]
for n in sizes:
    bundle_n = run_suite_maxcut(n=n, p_list=[1,2,3], graph_p=0.4, shots=512, seed=7)
    df_n = export_artifacts(bundle_n, f"qaoa_n{n}.csv", f"qaoa_n{n}.json")
    print(f"n={n} → avg approx ratio:",
          df_n["approx_ratio_mitig"].fillna(df_n["approx_ratio_plain"]).mean())
